# Lab 19b: Agents Inventory 

This notebook demonstrates how to list **all agents** using Azure ARM and data plane APIs. The goal is to showcase what happens behind the scenes.

## Agent Types

| Type | Source | How to List |
|------|--------|-------------|
| **Foundry Agents** | Created in AI Foundry portal | Data plane: `AIProjectClient.agents.list()` |
| **Custom Agents** | Registered via APIM | ARM: APIM APIs where `isAgent: true` |

The portal identifies custom agents by the **`isAgent: true`** flag on APIM APIs:

```json
{
  "properties": {
    "isAgent": true,
    "agent": {
      "id": "my-agent-id",
      "name": "my-agent"
    },
    "serviceUrl": "https://my-backend.com",
    "path": "my-agent-id"
  }
}
```

## Setup

In [13]:
from inventory_helpers import (
    list_all_projects, list_foundry_agents, list_custom_agents,
    list_connected_projects, is_agent_blocked,
    register_custom_agent, unregister_custom_agent,
    block_custom_agent, unblock_custom_agent,
    load_env_config,
    mask_resource_name, mask_subscription, mask_url
)

# Load config
config = load_env_config()
SUBSCRIPTION = config['subscription']
APIM_NAME = config['apim_name']
APIM_RG = config['apim_rg']


---

# Part 1: List Connected Foundry Projects

Lists projects connected to APIM via products (not all projects in subscription).

In [2]:
# Get projects connected to APIM via products (not all projects in subscription)
projects = list_connected_projects(SUBSCRIPTION, APIM_RG, APIM_NAME)

print(f"Found {len(projects)} connected projects.")

Found 4 connected projects.


---

# Part 2: List Foundry Agents

These are agents created natively in AI Foundry projects.

**API**: `AIProjectClient.agents.list()` (data plane)

In [3]:
all_foundry_agents = []

print("Listing Foundry agents per project:")
print("=" * 60)

for p in projects:
    account = p['accountName']
    project = p['projectName']
    
    agents = list_foundry_agents(account, project)
    all_foundry_agents.extend(agents)
    
    if agents:
        print(f"\n{mask_resource_name(account)}/{mask_resource_name(project)}:")
        for a in agents:
            print(f"  - {a.name} ({a.id[:8]}...)")

print(f"\nTotal Foundry agents: {len(all_foundry_agents)}")

Listing Foundry agents per project:

contoso-team-******/inventory-ai:
  - demoagent (demoagen...)
  - testcontosinventmodel (testcont...)
  - testcontosinventgpt4 (testcont...)

fabrikam-team-******/doc-******:
  - testfabrikdocstgrok3 (testfabr...)
  - testfabrikdocstgpt4o (testfabr...)

foundry-spoke-******/project-******:
  - iss-tracker (iss-trac...)
  - docs-expert (docs-exp...)
  - docs-expert-governed (docs-exp...)
  - policytestspoketestgpt41mini (policyte...)
  - model-router-agent (model-ro...)
  - hello-world-agent (hello-wo...)

woodgrove-team-******/risk-******:
  - testwoodgrriskagpt4 (testwood...)
  - testwoodgrriskaDeepSe (testwood...)
  - testwoodgrriskagrok3 (testwood...)

Total Foundry agents: 14


---

# Part 3: List Custom Agents

These are external agents registered via APIM (either through portal or our API).

**API**: `GET .../apis?api-version=2024-05-01` + filter `isAgent: true`

In [4]:
custom_agents = list_custom_agents(SUBSCRIPTION, APIM_RG, APIM_NAME)

print(f"Custom Agents ({len(custom_agents)}):")
print("=" * 60)

for a in custom_agents:
    print(f"\n  Name: {a.name}")
    print(f"  Agent ID: {a.agent_id}")
    print(f"  Backend URL: {a.backend_url}")
    print(f"  Endpoint: {mask_url(a.gateway_url)}/{a.agent_id}")
    print(f"  APIM API: {mask_resource_name(a.apim_api_name)}")

Custom Agents (0):


---

# Part 4: Combined Agent Inventory

Same view as the AI Foundry portal's "Agents" page.

In [14]:
print("COMPLETE AGENT INVENTORY")
print("=" * 80)
print(f"{'Name':<30} {'Source':<15} {'Project':<25} {'Status'}")
print("-" * 80)

# Foundry agents
for a in all_foundry_agents:
    print(f"{a.name:<30} {'Foundry':<15} {mask_resource_name(a.project):<25} Running")

# Custom agents
for a in custom_agents:
    print(f"{a.name:<30} {'Custom':<15} {'-':<25} Active")

print("-" * 80)
print(f"Total: {len(all_foundry_agents)} Foundry + {len(custom_agents)} Custom = {len(all_foundry_agents) + len(custom_agents)} agents")

COMPLETE AGENT INVENTORY
Name                           Source          Project                   Status
--------------------------------------------------------------------------------
demoagent                      Foundry         inventory-ai              Running
testcontosinventmodel          Foundry         inventory-ai              Running
testcontosinventgpt4           Foundry         inventory-ai              Running
testfabrikdocstgrok3           Foundry         doc-******                Running
testfabrikdocstgpt4o           Foundry         doc-******                Running
iss-tracker                    Foundry         project-******            Running
docs-expert                    Foundry         project-******            Running
docs-expert-governed           Foundry         project-******            Running
policytestspoketestgpt41mini   Foundry         project-******            Running
model-router-agent             Foundry         project-******            Running
hell

---

# Part 5: Register a Custom Agent

Register an external agent using the same method as the portal.

**API**: `PUT .../apis/{name}` with `isAgent: true`

In [6]:
# Configure agent
AGENT_NAME = "test-agent-cli"
AGENT_ID = "test-cli-001"  # This becomes the URL path
BACKEND_URL = "https://httpbin.org/anything"

print(f"Registering custom agent:")
print(f"  Name: {AGENT_NAME}")
print(f"  Agent ID: {AGENT_ID}")
print(f"  Backend: {BACKEND_URL}")

Registering custom agent:
  Name: test-agent-cli
  Agent ID: test-cli-001
  Backend: https://httpbin.org/anything


In [7]:
# Delete existing agent with same path if it exists (avoids path conflict)
for a in custom_agents:
    if a.agent_id == AGENT_ID:
        print(f"Found existing agent with path '{AGENT_ID}': {mask_resource_name(a.apim_api_name)}")
        delete_result = unregister_custom_agent(SUBSCRIPTION, APIM_RG, APIM_NAME, a.apim_api_name)
        print(f"Deleted: {delete_result}")
        break
else:
    print(f"No existing agent with path '{AGENT_ID}' found")

No existing agent with path 'test-cli-001' found


In [8]:
result = register_custom_agent(
    subscription=SUBSCRIPTION,
    apim_rg=APIM_RG,
    apim_name=APIM_NAME,
    agent_name=AGENT_NAME,
    agent_id=AGENT_ID,
    backend_url=BACKEND_URL,
    description="Test agent registered via CLI"
)

if 'error' in result:
    print(f"Error: {result['error']}")
else:
    print(f"\nAgent registered!")
    print(f"  APIM API: {mask_resource_name(result['apiName'])}")
    print(f"  Endpoint: {mask_url(result['agentEndpoint'])}")


Agent registered!
  APIM API: test-agent-cli-******
  Endpoint: https://foundry-apim-******.azure-api.net/test-cli-001


## Verify Registration

In [9]:
# Re-list custom agents
custom_agents = list_custom_agents(SUBSCRIPTION, APIM_RG, APIM_NAME)

print(f"Custom Agents ({len(custom_agents)}):")
for a in custom_agents:
    print(f"  - {a.name} (ID: {a.agent_id}, API: {mask_resource_name(a.apim_api_name)})")

Custom Agents (1):
  - test-agent-cli (ID: test-cli-001, API: test-agent-cli-******)


---

# Part 6: Agent Lifecycle (Block/Unblock)

In [10]:
# Block the agent
if 'apiName' in result:
    block_result = block_custom_agent(SUBSCRIPTION, APIM_RG, APIM_NAME, result['apiName'])
    print(f"Agent blocked: {block_result}")
else:
    print("No agent to block")

Agent blocked: {'success': True, 'status': 'blocked', 'apiName': 'test-agent-cli-0f6adb33'}


In [11]:
# Unblock the agent
if 'apiName' in result:
    unblock_result = unblock_custom_agent(SUBSCRIPTION, APIM_RG, APIM_NAME, result['apiName'])
    print(f"Agent unblocked: {unblock_result}")

Agent unblocked: {'success': True, 'status': 'active', 'apiName': 'test-agent-cli-0f6adb33'}


---

# Cleanup

In [12]:
# Uncomment to unregister the test agent
# if 'apiName' in result:
#     delete_result = unregister_custom_agent(SUBSCRIPTION, APIM_RG, APIM_NAME, result['apiName'])
#     print(f"Agent unregistered: {delete_result}")

print("Cleanup code is commented out")

Cleanup code is commented out


---

## Summary: APIs Used

### List Agents

| Agent Type | API | Method |
|------------|-----|--------|
| **Foundry** | `https://{account}.services.ai.azure.com/api/projects/{project}/agents` | Data plane via SDK |
| **Custom** | `GET .../Microsoft.ApiManagement/service/{apim}/apis` | ARM (filter `isAgent: true`) |

### Register Custom Agent

```http
PUT https://management.azure.com/subscriptions/{sub}/resourceGroups/{rg}/
    providers/Microsoft.ApiManagement/service/{apim}/apis/{name}
    ?api-version=2024-05-01

{
  "properties": {
    "displayName": "my-agent",
    "path": "agent-id",
    "serviceUrl": "https://my-backend.com",
    "isAgent": true,
    "agent": { "id": "agent-id", "name": "my-agent" }
  }
}
```

### Block/Unblock

```http
PUT .../apis/{name}/policies/policy

# Block: <return-response><set-status code="503"/></return-response>
# Unblock: <set-backend-service base-url="{url}"/>
```